# PICKO · Notebook 1 — Generate Training Data with Gemini

Synthesize `{query, tools, answers}` training examples for chosen scientific tools, with a **live
progress bar**, and end with **exactly N examples per tool**. Output is a JSONL that `needle finetune`
(and Notebook 2) consume directly.

**Flow:** pick tools → configure → *(optional)* broad generation → **force every tool to ≥ target**
→ **balance to exactly N/tool** → validate → chart.

> Requires `GEMINI_API_KEY` in `.env`. Kernel: **PICKO (.venv)**.

> **Setup:** this notebook imports `needle` (upstream Cactus package). Install it first — see the README (`pip install -e ./needle`).

### 1 · Setup

In [ ]:
import os, sys, json
ROOT = os.path.abspath("..")
if ROOT not in sys.path: sys.path.insert(0, ROOT)
envp = os.path.join(ROOT, ".env")
if os.path.exists(envp):
    for line in open(envp):
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1); os.environ.setdefault(k.strip(), v.strip())
assert os.environ.get("GEMINI_API_KEY"), "Set GEMINI_API_KEY in .env"

from scripts.tool_catalog import Catalog, family_of
import scripts.generate_picko_data as gp
import pandas as pd
from tqdm.auto import tqdm
cat = Catalog()
print(f"Catalog: {len(cat.tools)} tools across {len(cat.list_families())} families")

### 2 · Pick the tools to generate for  ⚠️ this defines EVERYTHING

`tools` here is the single source of truth: the forcing step (5) and the balancing step (6) both use
**exactly this set**. Whatever you want 120-each for must be in `SELECTION`.

- `dict()` → **ALL 75 tools** (every family) ← use this for "examples for all tools"
- `dict(one_per_family=True)` → the 11 one-per-product tools (great for **D3**)
- `dict(families=["arxiv", "pubmed"])` → whole families · `dict(categories=[...])` · `dict(names=[...])`

In [ ]:
SELECTION = dict()                         # <-- EDIT.  dict() = ALL 75 tools
tools, pools = cat.select_tools(**SELECTION)
TOOL_NAMES = [t["name"] for t in tools]
print(f"SELECTION -> {len(tools)} tools across {len(pools)} families")
pd.DataFrame([{"tool": t["name"], "family": family_of(t["name"]),
               "required": cat.params_of(t["name"])[0], "total_params": cat.params_of(t["name"])[1]}
              for t in tools])

### 3 · Configure

In [ ]:
TARGET_PER_TOOL = 130      # force every tool to at least this many (buffer above 120)
FINAL_PER_TOOL  = 120      # exact count per tool after balancing
WORKERS         = 3        # concurrency (the RPM gate below is the real throttle)
RPM             = 13       # Gemini free tier = 15 RPM; 13 leaves headroom for retries
BATCH_SIZE      = 15
DO_BROAD        = True     # quick broad pass first (diverse multi-tool contexts + negatives)
gp.set_rate_limit(RPM)     # <-- spaces ALL requests so we never exceed the RPM limit
print(f"rate limit: {RPM} req/min (~{60/RPM:.1f}s between requests)")
OUTPUT_JSONL    = os.path.join(ROOT, "data", "picko_gen.jsonl")   # raw (unbalanced) pool
BALANCED_JSONL  = os.path.join(ROOT, "data", "picko_balanced.jsonl")  # exactly N/tool -> finetune this
MODEL           = gp.g.MODEL

cp = gp.g.ClientPool(gp.g.make_clients())          # one Gemini client pool, reused below
seen, per_tool = gp._existing_state(OUTPUT_JSONL)  # resume-friendly (reads existing file)
print("resuming from", OUTPUT_JSONL, "| existing positives:", sum(per_tool.values()))

### 4 · (optional) Broad pass

Offers the whole selected set to Gemini and lets it choose which tool each query calls. Fast coverage
for the *prominent* tools, plus organic multi-tool contexts and abstention negatives. **Niche tools
stay under-covered here — that's fixed in step 5.**

In [ ]:
if DO_BROAD:
    gp._apply_patches(pools)
    goal = len(tools) * TARGET_PER_TOOL
    bar = tqdm(total=goal, desc="broad positives", unit="ex"); bar.update(min(sum(per_tool.values()), goal))
    def on_batch(n_new, n_pos, pt):
        bar.n = min(sum(pt.values()), goal); bar.refresh()
    need = max(0, goal - sum(per_tool.values()))
    new = gp.robust_generate(cp, MODEL, WORKERS, BATCH_SIZE, 300, seen, need, per_tool, on_batch)
    gp._append(OUTPUT_JSONL, new); bar.close()
    print("broad added", len(new))
else:
    print("skipped broad pass")

### 5 · Force every tool to ≥ TARGET  ⭐

The guarantee. For each tool still short, we offer **only that tool** with call-type locked to
`single`, so Gemini *must* write a query that calls it (no more starved niche tools). Distractor tools
are then re-injected into each example's context so training still sees a realistic multi-tool prompt.
Runs per tool with its own bar.

In [ ]:
under = [t for t in tools if per_tool[t["name"]] < TARGET_PER_TOOL]
print(f"{len(under)} / {len(tools)} tools under {TARGET_PER_TOOL} — forcing each:")
print("  ", [t["name"] for t in under])   # <- these are exactly the tools that will be generated
try:
    for t in tqdm(under, desc="tools"):
        need = TARGET_PER_TOOL - per_tool[t["name"]]
        b = tqdm(total=need, desc=t["name"], leave=False)
        cb = lambda n_new, n_pos, pt, b=b: (setattr(b, "n", min(n_pos, b.total)), b.refresh())
        new = gp.force_tool(cp, t, cat.tools, need, seen, per_tool, MODEL,
                            workers=WORKERS, batch_size=BATCH_SIZE, on_batch=cb)
        gp._append(OUTPUT_JSONL, new); b.close()
except gp.QuotaExhausted as e:
    print("\n⛔ STOPPED:", e)
    print("Everything generated so far is saved. Re-run this cell after the daily "
          "quota resets (or enable billing) to finish the rest.")
low = min((per_tool[t["name"]] for t in tools), default=0)
n_ready = sum(1 for t in tools if per_tool[t["name"]] >= FINAL_PER_TOOL)
print(f"lowest per-tool count now: {low} | tools >= {FINAL_PER_TOOL}: {n_ready}/{len(tools)}")


### 6 · Balance to EXACTLY N per tool

Randomly subsample every tool down to `FINAL_PER_TOOL` (and cap negatives) → a clean, balanced
training file. **Finetune this file.**

In [ ]:
# keep_tools=TOOL_NAMES scopes the output to YOUR selection, so tools that happen
# to be in the file but aren't in SELECTION are dropped (no confusing 'short' noise).
rep = gp.balance_dataset(OUTPUT_JSONL, BALANCED_JSONL, n_per_tool=FINAL_PER_TOOL,
                         max_negatives=FINAL_PER_TOOL, keep_tools=TOOL_NAMES)
print("balanced ->", BALANCED_JSONL)
print("total lines:", rep["total"], "| negatives:", rep["negatives"])
if rep["short"]:
    print("STILL SHORT (re-run step 5 to force these):", rep["short"])
else:
    print(f"every one of the {len(TOOL_NAMES)} selected tools has exactly {FINAL_PER_TOOL} examples ✅")
pd.Series(rep["per_tool"]).sort_values()

### 7 · Validate the balanced file

In [ ]:
import subprocess
print(subprocess.run([sys.executable, os.path.join(ROOT,"scripts","check_picko_data.py"), BALANCED_JSONL],
                     capture_output=True, text=True).stdout)

### 8 · Visualize coverage

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
def counts(path):
    pos = Counter(); neg = 0
    for l in open(path):
        ex = json.loads(l); ans = json.loads(ex["answers"])
        if ans:
            for a in ans:
                if a.get("name"): pos[a["name"]] += 1
        else: neg += 1
    return pos, neg
for path, title in [(OUTPUT_JSONL, "raw pool"), (BALANCED_JSONL, "balanced (finetune this)")]:
    pos, neg = counts(path); s = pos.most_common()
    plt.figure(figsize=(8, 0.4*len(s)+1))
    plt.barh([k for k,_ in s][::-1], [v for _,v in s][::-1], color="#4C72B0")
    plt.axvline(120, color="crimson", ls="--", label="min 120")
    plt.title(f"{title}: per-tool positives  (+{neg} negatives)"); plt.legend(); plt.tight_layout(); plt.show()